# 10 — Manipulation stratégique et règles à comité (C#)

**Navigation** : [<< ChoixSocial-09 (série Python)](ChoixSocial-09-Committees-STV-Monroe-ChamberlinCourant-Python.ipynb) · [<< ChoixSocial-04 (jumeau C# précédent)](ChoixSocial-04-Computational-Aggregation-SAT-Z3-CSharp.ipynb) | [Twin Python](ChoixSocial-10-Manipulation-Strategique-MultiWinner-Python.ipynb) | [Index](README.md)

*Port C# .NET du carnet Python : mesurer les déviations stratégiques sous cinq règles à comité, puis confronter une recherche symbolique à la force brute.*

## Ce que ce carnet démontre

1. Le témoin de l'Exemple 6.1 : la pluralité passe d'un comité `a` (utilité 0) à `b` (utilité 1) quand la votante 3 déclare `b > c > a`.
2. Un détecteur exhaustif de déviations gagnantes, où le gain est calculé sur les préférences vraies.
3. Les résultats des cinq règles sur les 216 profils (`n=3, m=3`) et les 13 824 profils (`n=3, m=4`).
4. Un contrôle Z3 sur le profil `n=4, m=5, k=2`, puis une concordance de 100 % entre Z3 et l'énumération brute sur 200 profils reproductibles.

## Prérequis et conventions

- Kernel `.net-csharp` (.NET 9) ; Z3 est chargé par le package NuGet `Microsoft.Z3`.
- Un bulletin est un ordre strict complet ; un comité est un ensemble trié.
- À égalité, les candidats puis les comités sont départagés par ordre ordinal croissant. L'utilité est `m - rang` du meilleur membre du comité, avec rang commençant à 1.
- Le STV utilise le quota de Droop, transfère les surplus et complète par les survivants classés par voix décroissantes puis ordre ordinal.

## Plan

| Section | Contenu |
|---|---|
| 1–2 | Modèle et témoin de référence |
| 3–4 | Règles à comité et détection |
| 5–7 | Énumérations exhaustives |
| 8 | Recherche symbolique Z3 et comparaison brute |
| 9 | Exercices |
| 10 | Conclusion |

In [1]:
#r "nuget: Microsoft.Z3, 4.12.2"

using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;
using Microsoft.Z3;

// Fix the numeric culture so tie-sensitive examples and printed results are reproducible.
CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.DefaultThreadCurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.DefaultThreadCurrentUICulture = CultureInfo.InvariantCulture;
Console.WriteLine("Configuration OK : manipulation stratégique à comité — C# .NET 9");

Installing Packages Microsoft.Z3

Configuration OK : manipulation stratégique à comité — C# .NET 9


## 1. Profil, rang et utilité *best-in-W*

Chaque bulletin ordonne les candidats du meilleur au pire. L'utilité d'un votant pour un comité est le score de Borda de son meilleur élu : `m - rang`. Cette mesure permet de comparer le résultat sincère et le résultat obtenu après une fausse déclaration, tout en évaluant les deux avec le bulletin vrai.

In [2]:
// Enumerate strict rankings and committee subsets in deterministic candidate order.
// Generate ordered voter profiles: voters are distinct, so row permutations remain distinct profiles.
// Plurality and Borda use ordinal tie-breaking; returned committees are sorted for stable comparison.
// Chamberlin-Courant maximizes each voter\x27s utility from their best-ranked committee member.
// Monroe searches voter assignments subject to balanced seat quotas.
// STV repeatedly elects a quota-qualified candidate or eliminates the lowest tally.
// Shared combinatorial helpers feed both the five voting rules and the exhaustive verifier.
static class Election
{
    public static List<string[]> Permutations(string[] items)
    {
        var result = new List<string[]>();
        void Visit(List<string> prefix, List<string> rest)
        {
            if (rest.Count == 0) { result.Add(prefix.ToArray()); return; }
            foreach (var item in rest.ToArray()) { var next = new List<string>(rest); next.Remove(item); var p = new List<string>(prefix) { item }; Visit(p, next); }
        }
        Visit(new List<string>(), items.ToList()); return result;
    }
    public static IEnumerable<string[]> Combinations(string[] items, int k, int start = 0)
    {
        if (k == 0) { yield return Array.Empty<string>(); yield break; }
        for (int i = start; i <= items.Length - k; i++) foreach (var tail in Combinations(items, k - 1, i + 1)) yield return new[] { items[i] }.Concat(tail).ToArray();
    }
    public static IEnumerable<string[][]> Profiles(List<string[]> ballots, int n)
    {
        var prefix = new List<string[]>();
        IEnumerable<string[][]> Visit()
        { if (prefix.Count == n) { yield return prefix.ToArray(); yield break; } foreach (var ballot in ballots) { prefix.Add(ballot); foreach (var p in Visit()) yield return p; prefix.RemoveAt(prefix.Count - 1); } }
        return Visit();
    }
    public static int Rank(string[] ballot, string candidate) => Array.IndexOf(ballot, candidate) + 1;
    public static int Utility(string[] trueBallot, string[] committee, int m) => committee.Length == 0 ? 0 : m - committee.Min(c => Rank(trueBallot, c));
    public static string Key(IEnumerable<string> committee) => string.Join("|", committee.OrderBy(c => c, StringComparer.Ordinal));
    // Compare sorted committees lexicographically; this same tie-break is encoded in Z3 below.
    public static int Compare(string[] a, string[] b)
    { for (int i = 0; i < Math.Min(a.Length, b.Length); i++) { int c = StringComparer.Ordinal.Compare(a[i], b[i]); if (c != 0) return c; } return a.Length.CompareTo(b.Length); }
    public static string[] Plurality(string[][] profile, int k)
    {
        var candidates = profile.SelectMany(x => x).Distinct().OrderBy(x => x, StringComparer.Ordinal).ToArray(); var votes = profile.GroupBy(x => x[0]).ToDictionary(g => g.Key, g => g.Count());
        return candidates.OrderByDescending(c => votes.GetValueOrDefault(c)).ThenBy(c => c, StringComparer.Ordinal).Take(k).OrderBy(c => c, StringComparer.Ordinal).ToArray();
    }
    public static string[] Borda(string[][] profile, int k)
    {
        var candidates = profile.SelectMany(x => x).Distinct().OrderBy(x => x, StringComparer.Ordinal).ToArray(); int m = candidates.Length; var scores = candidates.ToDictionary(c => c, c => profile.Sum(b => m - Rank(b, c)));
        return candidates.OrderByDescending(c => scores[c]).ThenBy(c => c, StringComparer.Ordinal).Take(k).OrderBy(c => c, StringComparer.Ordinal).ToArray();
    }
    public static string[] ChamberlinCourant(string[][] profile, int k)
    {
        int m = profile[0].Length; string[] best = null; int bestScore = int.MinValue;
        foreach (var w in Combinations(profile[0].OrderBy(c => c, StringComparer.Ordinal).ToArray(), k)) { int score = profile.Sum(b => Utility(b, w, m)); if (score > bestScore || (score == bestScore && (best == null || Compare(w, best) < 0))) { best = w; bestScore = score; } }
        return best ?? Array.Empty<string>();
    }
    public static string[] Monroe(string[][] profile, int k)
    {
        int n = profile.Length, m = profile[0].Length, low = n / k, high = (n + k - 1) / k; string[] best = null; int bestScore = int.MinValue; // Each committee member receives floor(n/k) or ceil(n/k) voters.
        foreach (var w in Combinations(profile[0].OrderBy(c => c, StringComparer.Ordinal).ToArray(), k))
        {
            int[] counts = new int[k]; int score = int.MinValue;
            void Assign(int voter, int total) { if (voter == n) { if (counts.All(c => c >= low)) score = Math.Max(score, total); return; } for (int j = 0; j < k; j++) if (counts[j] < high) { counts[j]++; Assign(voter + 1, total + m - Rank(profile[voter], w[j])); counts[j]--; } }
            Assign(0, 0); if (score > bestScore || (score == bestScore && score != int.MinValue && (best == null || Compare(w, best) < 0))) { best = w; bestScore = score; }
        }
        return best ?? Array.Empty<string>();
    }
    public static string[] Stv(string[][] profile, int k)
    {
        int n = profile.Length; var candidates = profile[0].OrderBy(c => c, StringComparer.Ordinal).ToList(); int quota = n / (k + 1) + 1; // Droop quota; ties use ordinal candidate order.
        var elected = new List<string>(); var alive = new HashSet<string>(candidates); var votes = candidates.ToDictionary(c => c, c => 0.0);
        void Recount() { foreach (var c in candidates) votes[c] = 0; foreach (var b in profile) { var first = b.FirstOrDefault(alive.Contains); if (first != null) votes[first] += 1; } }
        Recount();
        while (elected.Count < k)
        {
            var qualified = alive.Where(c => votes[c] >= quota).OrderByDescending(c => votes[c]).ThenBy(c => c, StringComparer.Ordinal).FirstOrDefault();
            if (qualified != null)
            {
                double tally = votes[qualified]; elected.Add(qualified); alive.Remove(qualified); double surplus = tally - quota;
                if (surplus > 0 && tally > 0) foreach (var b in profile) { bool seen = false; foreach (var c in b) { if (c == qualified) { seen = true; continue; } if (seen && alive.Contains(c) && !elected.Contains(c)) { votes[c] += surplus / tally; break; } } }
                votes[qualified] = quota; continue;
            }
            if (alive.Count <= k - elected.Count) break;
            var eliminated = alive.OrderBy(c => votes[c]).ThenBy(c => c, StringComparer.Ordinal).First();
            var holders = profile.Select(b => b.First(c => alive.Contains(c) || elected.Contains(c))).ToArray();
            alive.Remove(eliminated);
            for (int i = 0; i < profile.Length; i++) if (holders[i] == eliminated)
                foreach (var c in profile[i]) if (alive.Contains(c) && !elected.Contains(c)) { votes[c] += 1; break; }
            votes[eliminated] = 0;
        }
        elected.AddRange(alive.OrderByDescending(c => votes[c]).ThenBy(c => c, StringComparer.Ordinal).Take(k - elected.Count));
        if (elected.Count != k) throw new InvalidOperationException("Le STV doit élire exactement k candidats.");
        return elected.OrderBy(c => c, StringComparer.Ordinal).ToArray();
    }
    public static Dictionary<string, Func<string[][], int, string[]>> Rules = new() { ["Pluralité"] = Plurality, ["Borda"] = Borda, ["Chamberlin-Courant"] = ChamberlinCourant, ["Monroe"] = Monroe, ["STV"] = Stv };
    public static bool HasProfitableDeviation(string[][] profile, int k, Func<string[][], int, string[]> rule, List<string[]> ballots)
    {
        int m = profile[0].Length; var honest = rule(profile, k).OrderBy(c => c, StringComparer.Ordinal).ToArray();
        // Enumerate unilateral reports; evaluate both outcomes with the voter's true preferences.
        for (int i = 0; i < profile.Length; i++) foreach (var falseBallot in ballots) { if (profile[i].SequenceEqual(falseBallot)) continue; var changed = (string[][])profile.Clone(); changed[i] = falseBallot; var outcome = rule(changed, k); if (!honest.SequenceEqual(outcome) && Utility(profile[i], outcome, m) > Utility(profile[i], honest, m)) return true; }
        return false;
    }
    public static (int total, int manipulable) Enumerate(int n, string[] candidates, int k, Func<string[][], int, string[]> rule)
    { int total = 0, manipulable = 0; var ballots = Permutations(candidates); foreach (var p in Profiles(ballots, n)) { total++; if (HasProfitableDeviation(p, k, rule, ballots)) manipulable++; } return (total, manipulable); }
}
Console.WriteLine("Outils de scrutin prêts : pluralité, Borda, Chamberlin-Courant, Monroe, STV.");

Outils de scrutin prêts : pluralité, Borda, Chamberlin-Courant, Monroe, STV.


## 2. Le témoin de l'Exemple 6.1

Les préférences vraies sont `a > b > c`, `b > c > a` et `c > b > a`. À la pluralité, chaque candidat obtient un point ; le départage alphabétique donne `a`, le dernier choix de la votante 3. Si celle-ci annonce `b > c > a`, `b` gagne et son utilité vraie passe de 0 à 1.

In [3]:
// Each row is a true ranking; only voter 3 reports a different order in the second profile.
// Compare both committees with voter 3\x27s original ranking, not with the strategic report.
string[][] r61 = { new[] { "a", "b", "c" }, new[] { "b", "c", "a" }, new[] { "c", "b", "a" } };
var honest61 = Election.Plurality(r61, 1); var changed61 = (string[][])r61.Clone(); changed61[2] = new[] { "b", "c", "a" }; var manipulated61 = Election.Plurality(changed61, 1);
Console.WriteLine($"Points sincères : {string.Join(", ", r61.GroupBy(b => b[0]).OrderBy(g => g.Key).Select(g => g.Key + "=" + g.Count()))}");
Console.WriteLine($"Comité sincère {Election.Key(honest61)} ; utilité vraie v3 = {Election.Utility(r61[2], honest61, 3)}");
Console.WriteLine($"Après déclaration b>c>a : {Election.Key(manipulated61)} ; utilité vraie v3 = {Election.Utility(r61[2], manipulated61, 3)}");

Points sincères : a=1, b=1, c=1


Comité sincère a ; utilité vraie v3 = 0


Après déclaration b>c>a : b ; utilité vraie v3 = 1


### Lecture du résultat

La sortie attendue est `a=1, b=1, c=1`, comité sincère `a` avec utilité 0, puis comité `b` avec utilité 1. Le changement est un gain strict évalué avec le classement vrai de la votante, pas avec le faux bulletin.

## 3. Règles à comité

Les cinq règles utilisent une même signature `(profil, k) -> comité trié`. Pluralité et Borda classent les candidats par score ; Chamberlin-Courant maximise la somme des utilités *best-in-W* ; Monroe maximise le score Borda d'une affectation respectant les quotas `floor(n/k)` et `ceil(n/k)` ; le STV applique le quota de Droop et transfère les surplus. La cellule suivante compare leurs sorties sur un profil fixe.

In [4]:
// Repeated first choices create a visible plurality winner; k=2 exposes rule-specific committee choices.
string[][] demo = { new[] { "a", "b", "c" }, new[] { "b", "c", "a" }, new[] { "c", "a", "b" }, new[] { "b", "c", "a" } };
foreach (var rule in Election.Rules) Console.WriteLine($"{rule.Key,-20} k=1 {Election.Key(rule.Value(demo, 1)),-5} k=2 {Election.Key(rule.Value(demo, 2))}");

Pluralité            k=1 b     k=2 a|b


Borda                k=1 b     k=2 b|c


Chamberlin-Courant   k=1 b     k=2 a|b


Monroe               k=1 b     k=2 a|b


STV                  k=1 b     k=2 b|c


## 4. Détecter une déviation gagnante

Une déviation doit modifier l'ensemble élu et améliorer strictement l'utilité du votant concerné, mesurée sur ses préférences vraies. Le détecteur parcourt tous les faux bulletins et tous les votants ; il renvoie vrai dès qu'il trouve un témoin.

In [5]:
// All six strict rankings are possible reports; the detector tests whether any one improves true utility.
var ballot3 = Election.Permutations(new[] { "a", "b", "c" });
foreach (var rule in Election.Rules) Console.WriteLine($"{rule.Key,-20} témoin sur l'Exemple 6.1 : {Election.HasProfitableDeviation(r61, 1, rule.Value, ballot3)}");

Pluralité            témoin sur l'Exemple 6.1 : True


Borda                témoin sur l'Exemple 6.1 : False


Chamberlin-Courant   témoin sur l'Exemple 6.1 : False


Monroe               témoin sur l'Exemple 6.1 : False


STV                  témoin sur l'Exemple 6.1 : False


## 5. Énumération exhaustive : trois votants, trois candidats

Trois candidats admettent `3! = 6` bulletins, donc `6^3 = 216` profils. Pour `k=1` puis `k=2`, on compte les profils manipulables pour chacune des cinq règles ; un taux nul signifie uniquement qu'aucune déviation n'a été trouvée dans cet espace fini.

In [6]:
// With 3 candidates there are 6 ballots and therefore 6^3 = 216 ordered voter profiles.
// Run the same finite space for both committee sizes so the denominators are directly comparable.
var candidates3 = new[] { "a", "b", "c" };
Console.WriteLine("Profils exhaustifs : n=3, m=3, 216 profils");
foreach (var rule in Election.Rules)
{
    var one = Election.Enumerate(3, candidates3, 1, rule.Value); var two = Election.Enumerate(3, candidates3, 2, rule.Value);
    Console.WriteLine($"{rule.Key,-20} k=1 {one.manipulable,3}/{one.total} ; k=2 {two.manipulable,3}/{two.total}");
}

Profils exhaustifs : n=3, m=3, 216 profils


Pluralité            k=1  36/216 ; k=2   0/216


Borda                k=1  51/216 ; k=2  21/216


Chamberlin-Courant   k=1  51/216 ; k=2   0/216


Monroe               k=1  51/216 ; k=2   0/216


STV                  k=1  24/216 ; k=2   0/216


### Lecture du résultat

Valeurs de référence : pluralité `36/216` et `0/216`, Borda `51/216` et `21/216`, Chamberlin-Courant `51/216` et `0/216`, Monroe `51/216` et `0/216`, STV `24/216` et `0/216` pour `k=1` puis `k=2`. Ces nombres décrivent ces implémentations et cet espace exhaustif, pas une propriété générale des règles.

## 6. Élargir à quatre candidats

L'espace contient `24^3 = 13 824` profils. Cette enumeration permet de comparer les fréquences lorsque la taille du domaine augmente ; elle est volontairement exhaustive, sans échantillonnage. Le calcul peut durer plusieurs minutes selon la machine.

In [7]:
// Four candidates give 24 ballots and 24^3 = 13,824 ordered voter profiles.
// Exhaustively compare each rule for k=1 and k=2; no random sampling is involved.
var candidates4 = new[] { "a", "b", "c", "d" };
Console.WriteLine("Profils exhaustifs : n=3, m=4, 13824 profils");
foreach (var k in new[] { 1, 2 }) foreach (var rule in Election.Rules)
{ var count = Election.Enumerate(3, candidates4, k, rule.Value); Console.WriteLine($"k={k} {rule.Key,-20} {count.manipulable,5}/{count.total}"); }

Profils exhaustifs : n=3, m=4, 13824 profils


k=1 Pluralité             4068/13824


k=1 Borda                 7080/13824


k=1 Chamberlin-Courant    7080/13824


k=1 Monroe                7080/13824


k=1 STV                   2880/13824


k=2 Pluralité              864/13824


k=2 Borda                 3630/13824


k=2 Chamberlin-Courant    4512/13824


k=2 Monroe                4512/13824


k=2 STV                    864/13824


### Lecture du résultat

Références pour `m=4` : `k=1` pluralité 4068, Borda/Chamberlin-Courant/Monroe 7080 chacun, STV 2880 ; `k=2` pluralité 864, Borda 3630, Chamberlin-Courant 4512, Monroe 4512, STV 864 — chaque nombre est sur 13 824 profils. La hausse de manipulabilité pour certaines règles montre que le résultat dépend aussi de l'espace considéré.

## 7. Contrôle symbolique par Z3

Pour Chamberlin-Courant, le faux bulletin est représenté par une permutation de rangs Z3. Pour chaque comité possible, les contraintes imposent qu'il soit le maximum du score *best-in-W* après la déviation, avec le départage lexicographique. Le gain est calculé sur le bulletin vrai. Le solveur doit être recoupé avec le calcul direct : un encodage SAT incorrect peut donner une réponse cohérente mais fausse.

In [8]:
// Encode one voter's false report as a permutation of candidate ranks; other voters stay fixed.
static bool Z3HasDeviation(string[][] profile, int k, int voter)
{
    int n = profile.Length, m = profile[0].Length; var candidates = profile[0].OrderBy(c => c, StringComparer.Ordinal).ToArray();
    var honest = Election.ChamberlinCourant(profile, k); int honestUtility = Election.Utility(profile[voter], honest, m);
    using var ctx = new Context();
    var ranks = candidates.ToDictionary(c => c, c => (IntExpr)ctx.MkIntConst("rank_" + c));
    var solver = ctx.MkSolver(); foreach (var c in candidates) solver.Add(ctx.MkGe(ranks[c], ctx.MkInt(1)), ctx.MkLe(ranks[c], ctx.MkInt(m))); solver.Add(ctx.MkDistinct(ranks.Values.Cast<Expr>().ToArray()));
    // Utility is determined by the best-ranked committee member; all committees use the voter's symbolic ranks.
    ArithExpr UtilityExpr(string[] committee)
    { var bestRank = (IntExpr)ctx.MkIntConst("best_" + string.Join("_", committee)); foreach (var c in committee) solver.Add(ctx.MkLe(bestRank, ranks[c])); solver.Add(ctx.MkOr(committee.Select(c => (BoolExpr)ctx.MkEq(bestRank, ranks[c])).ToArray())); return ctx.MkSub(ctx.MkInt(m), bestRank); }
    // The deviating voter's utility is symbolic; the remaining voters' scores use their fixed true ballots.
    ArithExpr ScoreExpr(string[] committee)
    { ArithExpr total = ctx.MkInt(0); for (int v = 0; v < n; v++) total = ctx.MkAdd(total, v == voter ? UtilityExpr(committee) : ctx.MkInt(Election.Utility(profile[v], committee, m))); return total; }
    var committees = Election.Combinations(candidates, k).ToArray();
    foreach (var proposed in committees)
    {
        if (Election.Utility(profile[voter], proposed, m) <= honestUtility) continue; solver.Push();
        var proposedScore = ScoreExpr(proposed);
        // Enforce that the proposed committee wins, including the same ordinal tie-break as the direct rule.
        foreach (var other in committees) if (!proposed.SequenceEqual(other)) { var otherScore = ScoreExpr(other); solver.Add(Election.Compare(proposed, other) > 0 ? ctx.MkGt(proposedScore, otherScore) : ctx.MkGe(proposedScore, otherScore)); }
        bool sat = solver.Check() == Status.SATISFIABLE; solver.Pop(); if (sat) return true;
    }
    return false;
}

// Positive control from the Python twin: the first three voters are UNSAT; voter 4 has a profitable report.
string[][] control = { new[] { "d", "b", "e", "c", "a" }, new[] { "b", "d", "c", "a", "e" }, new[] { "d", "e", "a", "c", "b" }, new[] { "e", "d", "c", "b", "a" } };
Console.WriteLine($"Comité CC honnête : {Election.Key(Election.ChamberlinCourant(control, 2))}");
for (int voter = 0; voter < control.Length; voter++) Console.WriteLine($"v{voter + 1} : {(Z3HasDeviation(control, 2, voter) ? "SAT" : "UNSAT")}");

Comité CC honnête : b|d


v1 : UNSAT


v2 : UNSAT


v3 : UNSAT


v4 : SAT


## 8. Concordance Z3 / force brute sur 200 profils

Le contrôle positif doit donner le comité honnête `b,d`, les votants 1 à 3 sans déviation Z3, et le votant 4 avec une déviation profitable (`SAT`) — le même résultat que le carnet Python jumeau. On compare ensuite, sur 200 profils reproductibles de `n,m ∈ {3,4}`, l'existence d'une déviation déterminée par la force brute à celle détectée par Z3. Les deux méthodes doivent être d'accord sur chaque profil.

In [9]:
// Fixed seed: both methods receive each identical sampled profile.
var rng = new Random(20261020); int concordant = 0, brutePositive = 0, z3Positive = 0;
for (int sample = 0; sample < 200; sample++)
{
    int n = rng.Next(3, 5), m = rng.Next(3, 5); var candidates = Enumerable.Range(0, m).Select(i => ((char)('a' + i)).ToString()).ToArray(); var ballots = Election.Permutations(candidates);
    var profile = Enumerable.Range(0, n).Select(_ => ballots[rng.Next(ballots.Count)]).ToArray(); bool brute = Election.HasProfitableDeviation(profile, 2, Election.ChamberlinCourant, ballots);
    bool symbolic = false;
    for (int voter = 0; voter < n && !symbolic; voter++) symbolic = Z3HasDeviation(profile, 2, voter);
    if (brute) brutePositive++; if (symbolic) z3Positive++;
    if (brute != symbolic) throw new InvalidOperationException($"Divergence Z3/force brute au profil {sample} : force brute={brute}, Z3={symbolic}");
    concordant++;
}
// Mutation control: invert the fixed witness's Z3 verdict and require the parity check to detect it.
string[][] divergentWitness = { new[] { "d", "b", "e", "c", "a" }, new[] { "b", "d", "c", "a", "e" }, new[] { "d", "e", "a", "c", "b" }, new[] { "e", "d", "c", "b", "a" } };
var witnessBallots = Election.Permutations(divergentWitness[0]);
bool witnessBrute = Election.HasProfitableDeviation(divergentWitness, 2, Election.ChamberlinCourant, witnessBallots);
bool witnessZ3 = false; for (int voter = 0; voter < divergentWitness.Length && !witnessZ3; voter++) witnessZ3 = Z3HasDeviation(divergentWitness, 2, voter);
bool mutatedZ3 = !witnessZ3;
bool mismatchDetected = witnessBrute != mutatedZ3;
if (!witnessBrute || !witnessZ3 || !mismatchDetected || concordant != 200) throw new InvalidOperationException("Le témoin, son contrôle divergent ou la concordance complète a échoué.");
Console.WriteLine($"Force brute : {brutePositive} profils manipulables"); Console.WriteLine($"Z3 : {z3Positive} profils manipulables"); Console.WriteLine($"Concordance : {concordant}/200 ; écarts : {200 - concordant}"); Console.WriteLine($"Témoin fixe : force brute={witnessBrute}, Z3={witnessZ3}, mutation inversée détectée={mismatchDetected}");

Force brute : 32 profils manipulables


Z3 : 32 profils manipulables


Concordance : 200/200 ; écarts : 0


Témoin fixe : force brute=True, Z3=True, mutation inversée détectée=True


### Lecture du résultat

La concordance est totale : sur les 200 profils tirés, la force brute et Z3 détectent exactement les mêmes 32 profils manipulables — aucun écart. Le compte absolu (32) diffère de celui du carnet Python (39) pour une raison de générateur : la graine `20261020` est identique des deux côtés, mais `System.Random` (C#) et le Mersenne Twister de `random` (Python) produisent des séquences différentes, donc des échantillons de 200 profils différents. Ce qui doit être identique d'un jumeau à l'autre n'est pas le tirage mais l'accord des deux méthodes sur chaque profil tiré — ici `Concordance : 200/200 ; écarts : 0`, comme côté Python.

## 9. Exercices

Les trois exercices sont des stubs compilables, sans exception volontaire.

1. Écrire `AverageUtility` et comparer les trois comités de taille 2 sur `demo`.
2. Compter les profils manipulables de pluralité et de STV sur `n=3, m=3, k=1`, puis comparer leurs fréquences.
3. Adapter `HasProfitableDeviation` pour ne tester qu'un électeur désigné et lister ses déviations pour la votante 2 de l'Exemple 6.1.

In [10]:
// Exercice 1 — à compléter : calculer l'utilité moyenne d'un comité.
static double AverageUtility(string[][] profile, string[] committee)
{
    // TODO etudiant : ecrire AverageUtility (moyenne des Election.Utility des votants sur le profil, m = nombre de candidats du profil).
    Console.WriteLine("Exercice à compléter");
    return 0.0;
}
foreach (var w in Election.Combinations(new[] { "a", "b", "c" }, 2)) Console.WriteLine($"{Election.Key(w)} : {AverageUtility(demo, w)}");

Exercice à compléter


a|b : 0


Exercice à compléter


a|c : 0


Exercice à compléter


b|c : 0


In [11]:
// Exercice 2 — à compléter : comparer les fréquences de manipulation.
static (int, int) ManipulationFrequency(Func<string[][], int, string[]> rule, int n, string[] candidates, int k)
{
    // TODO etudiant : enumerer les profils (Election.Profiles x Election.Permutations) et compter ceux qui admettent une deviation profitable (Election.HasProfitableDeviation), puis comparer Plurality et Stv sur n = 3, m = 3, k = 1.
    Console.WriteLine("Exercice à compléter");
    return (0, 0);
}
Console.WriteLine(ManipulationFrequency(Election.Plurality, 3, candidates3, 1)); Console.WriteLine(ManipulationFrequency(Election.Stv, 3, candidates3, 1));

Exercice à compléter


(0, 0)


Exercice à compléter


(0, 0)


In [12]:
// Exercice 3 — à compléter : isoler les déviations d'un votant.
static List<string[]> VoterDeviations(string[][] profile, int voter, int k)
{
    // TODO etudiant : enumerer les bulletins voisins du votant designe et ne garder que les deviations qui changent le resultat et ameliorent son utilite (votante 2 de l'Exemple 6.1 : r61).
    Console.WriteLine("Exercice à compléter");
    return new List<string[]>();
}
Console.WriteLine(VoterDeviations(r61, 1, 1).Count);

Exercice à compléter


0


### Lecture des exercices

Les trois cellules affichent `Exercice à compléter` et restent exécutables avant le travail étudiant, conformément à C.1. Elles ne remplacent aucun calcul de démonstration du carnet.

## 10. Conclusion

Le port C# met en œuvre la pluralité, Borda, Chamberlin-Courant, Monroe et STV, puis mesure leurs manipulations sur deux espaces exhaustifs. Le témoin de l'Exemple 6.1 est reproduit avec son gain vrai de 0 à 1. Enfin, l'encodage Z3 est comparé à la force brute sur 200 profils : la sortie doit afficher `Concordance : 200/200 ; écarts : 0`. Ces résultats sont propres aux règles, conventions et domaines explicitement testés.

## Carnet jumeau

- [ChoixSocial-10-Manipulation-Strategique-MultiWinner-Python.ipynb](ChoixSocial-10-Manipulation-Strategique-MultiWinner-Python.ipynb) — énumérations de référence et formulation Z3 Python.
- [Index de la sous-série SocialChoice](README.md).